In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ieee-fraud-detection/sample_submission.csv
/kaggle/input/ieee-fraud-detection/test_identity.csv
/kaggle/input/ieee-fraud-detection/train_identity.csv
/kaggle/input/ieee-fraud-detection/test_transaction.csv
/kaggle/input/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install -q scikit-learn==1.3.2 imbalanced-learn==0.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.6/235.6 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nilearn 0.11.1 requires scikit-learn>=1.4.0, but you have scikit-learn 1.3.2 which is incompatible.
bigframes 1.36.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.


In [3]:
!pip install -q category_encoders

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OneHotEncoder
from category_encoders.woe import WOEEncoder

from imblearn.under_sampling import RandomUnderSampler

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from imblearn.pipeline import Pipeline  

import gc

from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score

In [5]:
train_identity = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_identity.csv")
train_transaction = pd.read_csv("/kaggle/input/ieee-fraud-detection/train_transaction.csv")

test_identity = pd.read_csv("/kaggle/input/ieee-fraud-detection/test_identity.csv")
test_transaction = pd.read_csv("/kaggle/input/ieee-fraud-detection/test_transaction.csv")

# Cleaning & Feature Engineering

In [6]:
# Merge transaction and identity
train = train_transaction.merge(train_identity, how='left', on='TransactionID')
test = test_transaction.merge(test_identity, how='left', on='TransactionID')

# Drop TransactionID (only keep if needed for identification)
test_ID = test['TransactionID']
train.drop('TransactionID', axis=1, inplace=True)
test.drop('TransactionID', axis=1, inplace=True)

# Set target
y_train = train['isFraud']
X_train = train.drop('isFraud', axis=1)

In [7]:
X_train.shape, y_train.shape

((590540, 432), (590540,))

In [8]:
# Get columns that are present in both train and test
common_columns = X_train.columns.intersection(test.columns)

# Keep only those columns in both datasets
X_train = X_train[common_columns]
test = test[common_columns]

In [9]:
num_cols = X_train.columns[X_train.dtypes != 'object'].tolist()
len(num_cols)

378

In [10]:
cat_cols = X_train.columns[X_train.dtypes == 'object'].tolist()
len(cat_cols)

16

In [11]:
cat_unique = X_train[cat_cols].nunique()

In [12]:
for df in [X_train]:
    df['TransactionDT_days'] = df['TransactionDT'] / (3600 * 24)
    df['TransactionDT_hours'] = df['TransactionDT'] / 3600
    df['Transaction_hour'] = (df['TransactionDT'] // 3600) % 24
    df['Transaction_day'] = (df['TransactionDT'] // (3600 * 24)) % 7

In [13]:
missing = X_train.isnull().mean() * 100
missing = missing.sort_values(ascending=False) 

In [14]:
# 1. Drop columns with >90% missing
drop_cols = missing[missing > 90].index.tolist()
X_train.drop(columns=drop_cols, inplace=True)
# X_test.drop(columns=drop_cols, inplace=True)

# 2. Fill 50-90% missing columns
cols_50_90 = missing[(missing > 50) & (missing <= 90)].index.tolist()
for col in cols_50_90:
    if X_train[col].dtype == 'object':
        X_train.loc[:, col] = X_train[col].fillna('missing')
    else:
        X_train.loc[:, col] = X_train[col].fillna(-999)

# 3. Fill 10-50% missing columns
cols_10_50 = missing[(missing > 10) & (missing <= 50)].index.tolist()
for col in cols_10_50:
    if X_train[col].dtype == 'object':
        X_train.loc[:, col] = X_train[col].fillna('missing')
    else:
        X_train.loc[:, col] = X_train[col].fillna(-999)

# 4. Fill <10% missing columns
cols_under_10 = missing[(missing > 0) & (missing <= 10)].index.tolist()
for col in cols_under_10:
    if X_train[col].dtype == 'object':
        mode = X_train[col].mode()[0]
        X_train.loc[:, col] = X_train[col].fillna(mode)
    else:
        median = X_train[col].median()
        X_train.loc[:, col] = X_train[col].fillna(median)


In [15]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

X_train_split = X_train_split.reset_index(drop=True)
X_val_split = X_val_split.reset_index(drop=True)

y_train_split = y_train_split.reset_index(drop=True)
y_val_split = y_val_split.reset_index(drop=True)

In [16]:
woe_columns = list(cat_unique[cat_unique > 3].index)
one_hot_columns = list(cat_unique[cat_unique <= 3].index)

In [17]:
ohe = OneHotEncoder(sparse=False, drop='first', handle_unknown='ignore')
ohe.fit(X_train_split[one_hot_columns])

X_train_ohe = pd.DataFrame(
    ohe.transform(X_train_split[one_hot_columns]),
    columns=ohe.get_feature_names_out(one_hot_columns),
    index=X_train_split.index
)

X_val_ohe = pd.DataFrame(
    ohe.transform(X_val_split[one_hot_columns]),
    columns=ohe.get_feature_names_out(one_hot_columns),
    index=X_val_split.index
)

# X_test_ohe = pd.DataFrame(
#     ohe.transform(X_test[one_hot_columns]),
#     columns=ohe.get_feature_names_out(one_hot_columns),
#     index=X_test.index
# )

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [18]:
woe = WOEEncoder()
woe.fit(X_train_split[woe_columns], y_train_split)

X_train_woe = pd.DataFrame(
    woe.transform(X_train_split[woe_columns]),
    columns=woe_columns,
    index=X_train_split.index
)

X_val_woe = pd.DataFrame(
    woe.transform(X_val_split[woe_columns]),
    columns=woe_columns,
    index=X_val_split.index
)

# X_test_woe = pd.DataFrame(
#     woe.transform(X_test[woe_columns]),
#     columns=woe_columns,
#     index=X_test.index
# )

In [19]:
# Drop encoded categorical columns from original data using .loc[] to avoid copying
X_train_split.drop(columns=woe_columns + one_hot_columns, inplace=True)
X_val_split.drop(columns=woe_columns + one_hot_columns, inplace=True)
# X_test.drop(columns=woe_columns + one_hot_columns, inplace=True)

# Combine everything using .loc[] for column selection
X_train_encoded = pd.concat([X_train_split, X_train_ohe, X_train_woe], axis=1)
X_val_encoded = pd.concat([X_val_split, X_val_ohe, X_val_woe], axis=1)
# X_test_encoded = pd.concat([X_test, X_test_ohe, X_test_woe], axis=1)

In [20]:
gc.collect()

0

# Feature Selection

In [21]:
# Step 1: Compute correlation matrix on training set
corr_matrix = X_train_encoded.corr().abs()

# Step 2: Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Step 3: Find feature pairs with correlation greater than the threshold
threshold = 0.95
high_corr_pairs = [
    (corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j])
    for i in range(len(corr_matrix.columns))
    for j in range(i+1, len(corr_matrix.columns))
    if corr_matrix.iloc[i, j] > threshold
]

# Step 4: Choose feature to drop (one with lower correlation with target)
features_to_drop = []

for feat1, feat2, _ in high_corr_pairs:
    corr1 = abs(X_train_encoded[feat1].corr(y_train))
    corr2 = abs(X_train_encoded[feat2].corr(y_train))
    
    # Drop feature with lower correlation to target
    if corr1 < corr2:
        features_to_drop.append(feat1)
    else:
        features_to_drop.append(feat2)

# Step 5: Remove duplicates
features_to_drop = list(set(features_to_drop))

In [22]:
len(features_to_drop)

276

In [23]:
len(X_train_encoded.columns)

407

In [24]:
# Final: Drop features from the training and test sets
X_train_filtered = X_train_encoded.drop(columns=features_to_drop)
X_val_filtered = X_val_encoded.drop(columns=features_to_drop)
#X_test_filtered = X_test_encoded.drop(columns=features_to_drop)

print(f"\nDropped {len(features_to_drop)} highly correlated features.")


Dropped 276 highly correlated features.


In [25]:
del corr_matrix

In [26]:
del train_transaction
del train_identity 
del test_transaction
del test_identity

In [27]:
del X_train_split
del X_val_split

In [28]:
del X_train
del y_train

# Training

In [56]:
# Define scalers and model
scalers = [
    StandardScaler(), 
    MinMaxScaler(), 
    None
]

model = {
    "XGBoost": {
        "model": XGBClassifier(eval_metric='logloss', random_state=42, scale_pos_weight=27.57),
        "params": {
            "scaler": scalers,
            "model__max_depth": [5, 7, 10],
            "model__learning_rate": [0.01, 0.1, 0.2]
        }
    }
}

# Cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Loop through models
for name, config in model.items():
    print(f"Training {name}...")
    
    # Define pipeline with placeholders
    pipeline = Pipeline([
        ("scaler", StandardScaler()),  
       # ("under", RandomUnderSampler(random_state=42)),
        ("model", config["model"])
    ])

    # Replace `scaler` param name properly in param_grid
    param_grid = config["params"]
    param_grid["scaler"] = scalers  # in pipeline, first step is named "scaler"

    # Grid search
    grid_search = GridSearchCV(
        pipeline,
        param_grid=config["params"],
        scoring='f1',  # Use 'f1', 'recall', or 'roc_auc' for imbalanced data
        cv=kfold,
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )

    grid_search.fit(X_train_filtered, y_train_split)

    print(f"Best parameters for {name}: {grid_search.best_params_}")
    print(f"Best cross-validation f1: {grid_search.best_score_:.4f}\n")


Training XGBoost...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best parameters for XGBoost: {'model__learning_rate': 0.2, 'model__max_depth': 10, 'scaler': StandardScaler()}
Best cross-validation f1: 0.5670



In [57]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
   # ('undersampler', RandomUnderSampler(random_state=42)),
    ('classifier', XGBClassifier(
        scale_pos_weight=27.57,
        random_state=42,
        eval_metric='logloss',
        learning_rate=0.2,
        max_depth=10
    ))
])

# Fit the pipeline on training data
pipeline.fit(X_train_filtered, y_train_split)

# Predict on train and validation
y_train_pred = pipeline.predict(X_train_filtered)
y_val_pred = pipeline.predict(X_val_filtered)
y_train_proba = pipeline.predict_proba(X_train_filtered)[:, 1]
y_val_proba = pipeline.predict_proba(X_val_filtered)[:, 1]

In [58]:
# Train metrics
train_f1 = f1_score(y_train_split, y_train_pred)
train_recall = recall_score(y_train_split, y_train_pred)
train_precision = precision_score(y_train_split, y_train_pred)
train_accuracy = accuracy_score(y_train_split, y_train_pred)
train_auc = roc_auc_score(y_train_split, y_train_proba)

# Validation metrics
val_f1 = f1_score(y_val_split, y_val_pred)
val_recall = recall_score(y_val_split, y_val_pred)
val_precision = precision_score(y_val_split, y_val_pred)
val_accuracy = accuracy_score(y_val_split, y_val_pred)
val_auc = roc_auc_score(y_val_split, y_val_proba)

In [59]:
print(f"Train F1 Score: {train_f1:.4f}")
print(f"Train Recall: {train_recall:.4f}")
print(f"Train Precision: {train_precision:.4f}")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Train AUC-ROC: {train_auc:.4f}")

Train F1 Score: 0.6944
Train Recall: 0.9581
Train Precision: 0.5445
Train Accuracy: 0.9705
Train AUC-ROC: 0.9954


In [60]:
print(f"Validation F1 Score: {val_f1:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation AUC-ROC: {val_auc:.4f}")

Validation F1 Score: 0.5805
Validation Recall: 0.8011
Validation Precision: 0.4551
Validation Accuracy: 0.9595
Validation AUC-ROC: 0.9569


# MLflow Logging

In [30]:
!pip install -q dagshub mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 43.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 77.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.0/700.0 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. Th

In [34]:
import dagshub
import mlflow

In [35]:
dagshub.init(repo_owner='mrekh21', repo_name='Fraud_Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=e2caf067-9d05-4ec4-bd99-f6413b68d5cd&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=184b270a6abd3b09d86822686b8c05d3baf576459d4a6d28d9296dbe20991670




Accessing as mrekh21

Initialized MLflow to track repo "mrekh21/Fraud_Detection"

Repository mrekh21/Fraud_Detection initialized!

In [62]:
experiment_name = "XGBoost_Training"
run_name = "cross_validation_6"

# Set the experiment (creates if it doesn't exist)
mlflow.set_experiment(experiment_name)

# Start an MLflow run with a specific name
with mlflow.start_run(run_name=run_name):

    # Log hyperparameters
    mlflow.log_params({
        'model' : "XGBClassifier",
        'scaler': "StandardScaler",
       # 'sampling' : 'RandomUnderSampler',
        'scale_pos_weight': 27.57,
        'random_state' : 42,
        'eval_metric' : 'logloss',
        'learning_rate' : 0.2,
        'max_depth' : 10
    })

    # Log performance metric
    mlflow.log_metric("train_f1", train_f1)
    mlflow.log_metric("train_recall", train_recall)
    mlflow.log_metric("train_precision", train_precision)
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("train_auc_roc", train_auc)
    
    mlflow.log_metric("val_f1", val_f1)
    mlflow.log_metric("val_recall", val_recall)
    mlflow.log_metric("val_precision", val_precision)
    mlflow.log_metric("val_accuracy", val_accuracy)
    mlflow.log_metric("val_auc_roc", val_auc)


#    mlflow.set_tag("Description", "Training XGBoost with undersampling and StandardScaler. train_transaction + train_indentity, OneHot + WOE, missing values handled with 'missing'/mode or -999/median")
    mlflow.set_tag("Description", "Training XGBoost with StandardScaler and positive weight scaling. train_transaction + train_indentity, OneHot + WOE, missing values handled with 'missing'/mode or -999/median, correlation filter(threshold=0.95)")

print("Pipeline logged successfully!")

🏃 View run cross_validation_6 at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/0/runs/dc090a48944846e89ab7a848f9b29425
🧪 View experiment at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/0
Pipeline logged successfully!


# Final Pipeline 

In [32]:
from sklearn.base import BaseEstimator, TransformerMixin


class CustomPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, missing, one_hot_columns, woe_columns, features_to_drop, common_columns):
        self.missing = missing
        self.one_hot_columns = one_hot_columns
        self.woe_columns = woe_columns
        self.features_to_drop = features_to_drop
        self.common_columns = common_columns
        self.ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
        self.woe = WOEEncoder()  # Assuming WOEEncoder is properly defined elsewhere

    def fit(self, X, y=None):
        X = X.copy()
        X = X[[col for col in self.common_columns if col in X.columns]]

        # Drop TransactionID
        if 'TransactionID' in X.columns:
            X.drop(columns=['TransactionID'], inplace=True)

        # Time features
        X['TransactionDT_days'] = X['TransactionDT'] / (3600 * 24)
        X['TransactionDT_hours'] = X['TransactionDT'] / 3600
        X['Transaction_hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_day'] = (X['TransactionDT'] // (3600 * 24)) % 7

        # Drop high missing columns
        drop_cols = self.missing[self.missing > 90].index.tolist()
        self.drop_cols_ = [col for col in drop_cols if col in X.columns]
        X.drop(columns=self.drop_cols_, inplace=True, errors='ignore')

        # Collect missing categories
        self.cols_50_90 = [col for col in self.missing[(self.missing > 50) & (self.missing <= 90)].index if col in X.columns]
        self.cols_10_50 = [col for col in self.missing[(self.missing > 10) & (self.missing <= 50)].index if col in X.columns]
        self.cols_under_10 = [col for col in self.missing[(self.missing > 0) & (self.missing <= 10)].index if col in X.columns]

        # Store fill values
        self.fill_modes = {col: X[col].mode()[0] for col in self.cols_under_10 if X[col].dtype == 'object'}
        self.fill_medians = {col: X[col].median() for col in self.cols_under_10 if X[col].dtype != 'object'}

        # Fit encoders
        self.ohe.fit(X[self.one_hot_columns])
        self.woe.fit(X[self.woe_columns], y)

        return self

    def transform(self, X):
        X = X.copy()
        X = X[[col for col in self.common_columns if col in X.columns]]

        # Drop TransactionID
        if 'TransactionID' in X.columns:
            X.drop(columns=['TransactionID'], inplace=True)

        # Time features
        X['TransactionDT_days'] = X['TransactionDT'] / (3600 * 24)
        X['TransactionDT_hours'] = X['TransactionDT'] / 3600
        X['Transaction_hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_day'] = (X['TransactionDT'] // (3600 * 24)) % 7

        # Drop high missing columns
        X.drop(columns=self.drop_cols_, inplace=True, errors='ignore')

        # Fill missing values
        for col in self.cols_50_90 + self.cols_10_50:
            if col in X.columns:
                X[col] = X[col].fillna('missing' if X[col].dtype == 'object' else -999)

        for col in self.cols_under_10:
            if col in X.columns:
                if X[col].dtype == 'object':
                    X[col] = X[col].fillna(self.fill_modes.get(col, 'missing'))
                else:
                    X[col] = X[col].fillna(self.fill_medians.get(col, -999))

         # fill any remaining null values
        X.fillna({'object': 'missing', 'numeric': -999}, inplace=True)

        # Encode
        X_ohe = pd.DataFrame(self.ohe.transform(X[self.one_hot_columns]),
                             columns=self.ohe.get_feature_names_out(self.one_hot_columns),
                             index=X.index)
        X_woe = pd.DataFrame(self.woe.transform(X[self.woe_columns]),
                             columns=self.woe_columns,
                             index=X.index)

        # Drop original categorical columns
        X.drop(columns=self.one_hot_columns + self.woe_columns, inplace=True, errors='ignore')

        # Merge all
        X_final = pd.concat([X, X_ohe, X_woe], axis=1)

        # Drop final features
        X_final.drop(columns=self.features_to_drop, inplace=True, errors='ignore')

        # Ensure all data is numeric
        X_final = X_final.apply(pd.to_numeric, errors='coerce')

        return X_final


In [122]:
pipeline = Pipeline([
    ('preprocessor', CustomPreprocessor(
        missing=missing,
        one_hot_columns=one_hot_columns,
        woe_columns=woe_columns,
        features_to_drop=features_to_drop,
        common_columns=common_columns
    )),
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(
        scale_pos_weight=27.57,
        random_state=42,
        eval_metric='logloss',
        learning_rate=0.1,
        max_depth=10,
        n_estimators=300
        # colsample_bytree=0.8,
        # reg_alpha=1,
        # reg_lambda=1
    ))
])


In [36]:
X_train = train.drop(columns=['isFraud'])
y_train = train['isFraud']

# Split into train/validation sets with stratification
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Reset indices
X_train_split = X_train_split.reset_index(drop=True)
X_val_split = X_val_split.reset_index(drop=True)
y_train_split = y_train_split.reset_index(drop=True)
y_val_split = y_val_split.reset_index(drop=True)

In [37]:
# Fit the pipeline on training data
pipeline.fit(X_train_split, y_train_split)

# Predict on train and validation
y_train_pred = pipeline.predict(X_train_split)
y_val_pred = pipeline.predict(X_val_split)

y_train_proba = pipeline.predict_proba(X_train_split)[:, 1]
y_val_proba = pipeline.predict_proba(X_val_split)[:, 1]

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.261504 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15997
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 124
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [38]:
# Train metrics
train_f1 = f1_score(y_train_split, y_train_pred)
train_recall = recall_score(y_train_split, y_train_pred)
train_precision = precision_score(y_train_split, y_train_pred)
train_accuracy = accuracy_score(y_train_split, y_train_pred)
train_auc = roc_auc_score(y_train_split, y_train_proba)

# Validation metrics
val_f1 = f1_score(y_val_split, y_val_pred)
val_recall = recall_score(y_val_split, y_val_pred)
val_precision = precision_score(y_val_split, y_val_pred)
val_accuracy = accuracy_score(y_val_split, y_val_pred)
val_auc = roc_auc_score(y_val_split, y_val_proba)

In [39]:
print(f"Train F1 Score: {train_f1:.4f}")
print(f"Train Recall: {train_recall:.4f}")
print(f"Train Precision: {train_precision:.4f}")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Train AUC-ROC: {train_auc:.4f}")

Train F1 Score: 0.6354
Train Recall: 0.9480
Train Precision: 0.4778
Train Accuracy: 0.9619
Train AUC-ROC: 0.9929


In [40]:
print(f"Validation F1 Score: {val_f1:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation AUC-ROC: {val_auc:.4f}")

Validation F1 Score: 0.5506
Validation Recall: 0.8178
Validation Precision: 0.4150
Validation Accuracy: 0.9533
Validation AUC-ROC: 0.9584


In [41]:
test_preds_proba = pipeline.predict_proba(test)[:, 1]

# Prepare the submission dataframe
submission = pd.DataFrame({
    "TransactionID": test_ID,
    "isFraud": test_preds_proba
})

# Save to CSV
submission.to_csv("submission.csv", index=False)


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [33]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

xgb_model = XGBClassifier(
    scale_pos_weight=27.57,
    random_state=42,
    eval_metric='logloss',
    learning_rate=0.1,
    max_depth=10,
    n_estimators=300
)

catboost_model = CatBoostClassifier(
    scale_pos_weight=27.57,
    random_state=42,
    verbose=0,
    learning_rate=0.1,
    depth=10,
    iterations=300,
    loss_function='Logloss'
)

lightgbm_model = LGBMClassifier(
    scale_pos_weight=27.57,
    random_state=42,
    learning_rate=0.1,
    max_depth=10,
    n_estimators=300,
    objective='binary'
)


In [34]:
from sklearn.ensemble import VotingClassifier

ensemble_model = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('catboost', catboost_model),
        ('lightgbm', lightgbm_model)
    ],
    voting='soft'           # average predicted probabilities 
)


In [35]:
pipeline = Pipeline([
    ('preprocessor', CustomPreprocessor(
        missing=missing,
        one_hot_columns=one_hot_columns,
        woe_columns=woe_columns,
        features_to_drop=features_to_drop,
        common_columns=common_columns
    )),
    ('scaler', StandardScaler()),
    ('classifier', ensemble_model)
])

In [ ]:
import shap
import matplotlib.pyplot as plt

# Fit your pipeline
pipeline.fit(X_train_split, y_train_split)

# Create SHAP explainer for the trained classifier
explainer = shap.Explainer(pipeline.named_steps['classifier'], X_train_split)

# Compute SHAP values for the training data
shap_values = explainer(X_train_split)

# Summary plot for feature importance
shap.summary_plot(shap_values, X_train_split)

# You can also visualize for a specific instance (e.g., the first one)
shap.force_plot(shap_values[0])

# For individual prediction analysis:
shap.initjs()
shap.force_plot(shap_values[0])

# Optionally, use a dependence plot to analyze feature interactions
shap.dependence_plot("feature_name", shap_values, X_train_split)


Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] during transform. These unknown categories will be encoded as all zeros


# Final Model Logging

In [36]:
from sklearn.base import BaseEstimator, TransformerMixin
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier


class CustomPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, missing, one_hot_columns, woe_columns, features_to_drop, common_columns):
        self.missing = missing
        self.one_hot_columns = one_hot_columns
        self.woe_columns = woe_columns
        self.features_to_drop = features_to_drop
        self.common_columns = common_columns
        self.ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
        self.woe = WOEEncoder()  

    def fit(self, X, y=None):
        X = X.copy()
        X = X[[col for col in self.common_columns if col in X.columns]]

        # Drop TransactionID
        if 'TransactionID' in X.columns:
            X.drop(columns=['TransactionID'], inplace=True)

        # Time features
        X['TransactionDT_days'] = X['TransactionDT'] / (3600 * 24)
        X['TransactionDT_hours'] = X['TransactionDT'] / 3600
        X['Transaction_hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_day'] = (X['TransactionDT'] // (3600 * 24)) % 7

        # Drop high missing columns
        drop_cols = self.missing[self.missing > 90].index.tolist()
        self.drop_cols_ = [col for col in drop_cols if col in X.columns]
        X.drop(columns=self.drop_cols_, inplace=True, errors='ignore')

        # Collect missing categories
        self.cols_50_90 = [col for col in self.missing[(self.missing > 50) & (self.missing <= 90)].index if col in X.columns]
        self.cols_10_50 = [col for col in self.missing[(self.missing > 10) & (self.missing <= 50)].index if col in X.columns]
        self.cols_under_10 = [col for col in self.missing[(self.missing > 0) & (self.missing <= 10)].index if col in X.columns]

        # Store fill values
        self.fill_modes = {col: X[col].mode()[0] for col in self.cols_under_10 if X[col].dtype == 'object'}
        self.fill_medians = {col: X[col].median() for col in self.cols_under_10 if X[col].dtype != 'object'}

        # Fit encoders
        self.ohe.fit(X[self.one_hot_columns])
        self.woe.fit(X[self.woe_columns], y)

        return self

    def transform(self, X):
        X = X.copy()
        X = X[[col for col in self.common_columns if col in X.columns]]

        # Drop TransactionID
        if 'TransactionID' in X.columns:
            X.drop(columns=['TransactionID'], inplace=True)

        # Time features
        X['TransactionDT_days'] = X['TransactionDT'] / (3600 * 24)
        X['TransactionDT_hours'] = X['TransactionDT'] / 3600
        X['Transaction_hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_day'] = (X['TransactionDT'] // (3600 * 24)) % 7

        # Drop high missing columns
        X.drop(columns=self.drop_cols_, inplace=True, errors='ignore')

        # Fill missing values
        for col in self.cols_50_90 + self.cols_10_50:
            if col in X.columns:
                X[col] = X[col].fillna('missing' if X[col].dtype == 'object' else -999)

        for col in self.cols_under_10:
            if col in X.columns:
                if X[col].dtype == 'object':
                    X[col] = X[col].fillna(self.fill_modes.get(col, 'missing'))
                else:
                    X[col] = X[col].fillna(self.fill_medians.get(col, -999))

         # fill any remaining null values
        X.fillna({'object': 'missing', 'numeric': -999}, inplace=True)

        # Encode
        X_ohe = pd.DataFrame(self.ohe.transform(X[self.one_hot_columns]),
                             columns=self.ohe.get_feature_names_out(self.one_hot_columns),
                             index=X.index)
        X_woe = pd.DataFrame(self.woe.transform(X[self.woe_columns]),
                             columns=self.woe_columns,
                             index=X.index)

        # Drop original categorical columns
        X.drop(columns=self.one_hot_columns + self.woe_columns, inplace=True, errors='ignore')

        # Merge all
        X_final = pd.concat([X, X_ohe, X_woe], axis=1)

        # Drop final features
        X_final.drop(columns=self.features_to_drop, inplace=True, errors='ignore')

        # Ensure all data is numeric
        X_final = X_final.apply(pd.to_numeric, errors='coerce')

        return X_final





xgb_model = XGBClassifier(
    scale_pos_weight=27.57,
    random_state=42,
    eval_metric='logloss',
    learning_rate=0.1,
    max_depth=10,
    n_estimators=300
)

catboost_model = CatBoostClassifier(
    scale_pos_weight=27.57,
    random_state=42,
    verbose=0,
    learning_rate=0.1,
    depth=10,
    iterations=300,
    loss_function='Logloss'
)

lightgbm_model = LGBMClassifier(
    scale_pos_weight=27.57,
    random_state=42,
    learning_rate=0.1,
    max_depth=10,
    n_estimators=300,
    objective='binary'
)



ensemble_model = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('catboost', catboost_model),
        ('lightgbm', lightgbm_model)
    ],
    voting='soft'           # average predicted probabilities 
)


pipeline = Pipeline([
    ('preprocessor', CustomPreprocessor(
        missing=missing,
        one_hot_columns=one_hot_columns,
        woe_columns=woe_columns,
        features_to_drop=features_to_drop,
        common_columns=common_columns
    )),
    ('scaler', StandardScaler()),
    ('classifier', ensemble_model)
])


In [37]:
X_train = train.drop(columns=['isFraud'])
y_train = train['isFraud']

# Split into train/validation sets with stratification
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Reset indices
X_train_split = X_train_split.reset_index(drop=True)
X_val_split = X_val_split.reset_index(drop=True)
y_train_split = y_train_split.reset_index(drop=True)
y_val_split = y_val_split.reset_index(drop=True)

In [41]:
experiment_name = "XGBoost_Training"
run_name = "Final_Pipeline"

# Set the experiment (creates if it doesn't exist)
mlflow.set_experiment(experiment_name)

# Start an MLflow run with a specific name
with mlflow.start_run(run_name=run_name):
    
    # Fit the pipeline on training data
    pipeline.fit(X_train_split, y_train_split)
    
     # Log model
    mlflow.sklearn.log_model(pipeline, 
                             artifact_path="final_model", 
                             registered_model_name="Fraud_Detection_Best_Model",
                            input_example=X_train_split[:5],  
                            signature=mlflow.models.infer_signature(X_train_split, y_train_pred))  
    
    # Predict on train and validation
    y_train_pred = pipeline.predict(X_train_split)
    y_val_pred = pipeline.predict(X_val_split)
    
    y_train_proba = pipeline.predict_proba(X_train_split)[:, 1]
    y_val_proba = pipeline.predict_proba(X_val_split)[:, 1]


    # Train metrics
    train_f1 = f1_score(y_train_split, y_train_pred)
    train_recall = recall_score(y_train_split, y_train_pred)
    train_precision = precision_score(y_train_split, y_train_pred)
    train_accuracy = accuracy_score(y_train_split, y_train_pred)
    train_auc = roc_auc_score(y_train_split, y_train_proba)
    
    # Validation metrics
    val_f1 = f1_score(y_val_split, y_val_pred)
    val_recall = recall_score(y_val_split, y_val_pred)
    val_precision = precision_score(y_val_split, y_val_pred)
    val_accuracy = accuracy_score(y_val_split, y_val_pred)
    val_auc = roc_auc_score(y_val_split, y_val_proba)


    # Log hyperparameters
    mlflow.log_params({
        'scaler': "StandardScaler",
        "classifier": "VotingClassifier",
        'xgb_scale_pos_weight': 27.57,
        'catboost_scale_pos_weight': 27.57,
        'loghtgbm_scale_pos_weight': 27.57,
        'xgb_random_state' : 42,
        'catboost_random_state' : 42,
        'loghtgbm_random_state' : 42,
        'xgb_eval_metric' : 'logloss',
        'catboost_loss_function': 'Logloss',
        'lightgbm_objective': 'binary',
        'xgb_learning_rate' : 0.1,
        'catboost_learning_rate' : 0.1,
        'lightgbm_learning_rate' : 0.1,
        'xgb_max_depth' : 10,
        'catboost_max_depth' : 10,
        'lightgbm_max_depth' : 10,
        'xgb_n_estimators': 300,
        'lightgbm_n_estimators': 300,
        'catboost_iterations': 300
    })

    mlflow.log_param("one_hot_columns", len(one_hot_columns))
    mlflow.log_param("woe_columns", len(woe_columns))

    mlflow.log_param("categroical_fill_1", "missing")
    mlflow.log_param("categroical_fill_2", "mode()")
    mlflow.log_param("numerical_fill_1", -999)
    mlflow.log_param("numerical_fill_2", "median()")
    mlflow.log_param("corr_filter_threshold", 0.95)

    # Log performance metric
    mlflow.log_metric("train_f1", train_f1)
    mlflow.log_metric("train_recall", train_recall)
    mlflow.log_metric("train_precision", train_precision)
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("train_auc_roc", train_auc)
    
    mlflow.log_metric("val_f1", val_f1)
    mlflow.log_metric("val_recall", val_recall)
    mlflow.log_metric("val_precision", val_precision)
    mlflow.log_metric("val_accuracy", val_accuracy)
    mlflow.log_metric("val_auc_roc", val_auc)



    # End the MLflow run
    mlflow.end_run()

print(f"Train F1 Score: {train_f1:.4f}")
print(f"Train Recall: {train_recall:.4f}")
print(f"Train Precision: {train_precision:.4f}")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Train AUC-ROC: {train_auc:.4f}")

print(f"Validation F1 Score: {val_f1:.4f}")
print(f"Validation Recall: {val_recall:.4f}")
print(f"Validation Precision: {val_precision:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation AUC-ROC: {val_auc:.4f}")

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.253885 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 15997
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 124
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101


/usr/local/lib/python3.11/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(

🏃 View run Final_Pipeline at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/0/runs/40343c4e9aef4490b3f33af22ee47e22
🧪 View experiment at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/0
Train F1 Score: 0.6354
Train Recall: 0.9480
Train Precision: 0.4778
Train Accuracy: 0.9619
Train AUC-ROC: 0.9929
Validation F1 Score: 0.5506
Validation Recall: 0.8178
Validation Precision: 0.4150
Validation Accuracy: 0.9533
Validation AUC-ROC: 0.9584


In [45]:
experiment_name = "XGBoost_Training"
run_name = "XGBoost_Cleaning"

# Set the experiment (creates if it doesn't exist)
mlflow.set_experiment(experiment_name)

# Start an MLflow run with a specific name
with mlflow.start_run(run_name=run_name):

    mlflow.log_param("one_hot_columns", len(one_hot_columns))
    mlflow.log_param("woe_columns", len(woe_columns))

    mlflow.log_param("missing_cat_fill_1", "missing")
    mlflow.log_param("missing_cat_fill_2", "mode()")
    mlflow.log_param("missing_num_fill_1", -999)
    mlflow.log_param("missing_num_fill_2", "median()")

    mlflow.log_param("data_balancing", 'scale_pos_weight')


    # End the MLflow run
    mlflow.end_run()



🏃 View run LogisticRegression_Cleaning at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/3/runs/5227cd1e549e4ba0b1c5a11d99034e2c
🧪 View experiment at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/3


In [49]:
experiment_name = "XGBoost_Training"
run_name = "XGBoost_Feature_Selection"

# Set the experiment (creates if it doesn't exist)
mlflow.set_experiment(experiment_name)

# Start an MLflow run with a specific name
with mlflow.start_run(run_name=run_name):

    mlflow.log_param("corr_threshold", 0.95)
    

    # End the MLflow run
    mlflow.end_run()



🏃 View run LogisticRegression_Feature_Selection at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/3/runs/8e6fb974a0154912a3c4392fcfd46691
🧪 View experiment at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/3


In [53]:
experiment_name = "XGBoost_Training"
run_name = "XGBoost_Feature_Engineering"

# Set the experiment (creates if it doesn't exist)
mlflow.set_experiment(experiment_name)

# Start an MLflow run with a specific name
with mlflow.start_run(run_name=run_name):
    
    mlflow.log_param("num_scaling_methods", "StandardScaler, MinMax")
    mlflow.log_param("cat_encoding", "WOE, OneHotEncoding")
    mlflow.log_param("new_features", "TransactionDT_days, TransactionDT_hours, Transaction_hour, Transaction_day")


🏃 View run LogisticRegression_Feature_Engineering at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/3/runs/8d96d23897284a03a3664cff84b93666
🧪 View experiment at: https://dagshub.com/mrekh21/Fraud_Detection.mlflow/#/experiments/3
